# IBRD Loan Data Cleaning & Feature Engineering

This notebook converts the raw World Bank IBRD loan snapshot into a clean silver dataset for portfolio analytics and risk reporting.

The workflow is intentionally non-destructive: raw values are inspected, normalized, and then saved to `data/processed/ibrd_clean.csv` for downstream use.

In [8]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

candidate_roots = [Path.cwd().resolve(), Path.cwd().resolve().parent]
project_root = next(
    (root for root in candidate_roots if (root / 'data' / 'raw' / 'ibrd_statement_of_loans_and_guarantees_latest_available_snapshot_09-09-2026.csv').exists()),
    Path('/home/rigii/ATA').resolve(),
)
raw_path = project_root / 'data' / 'raw' / 'ibrd_statement_of_loans_and_guarantees_latest_available_snapshot_09-09-2026.csv'
processed_path = project_root / 'data' / 'processed' / 'ibrd_clean.csv'
processed_path.parent.mkdir(parents=True, exist_ok=True)

print(f'Project root: {project_root}')
print(f'Raw data path: {raw_path}')
print(f'Output path: {processed_path}')


Project root: /home/rigii/ATA
Raw data path: /home/rigii/ATA/data/raw/ibrd_statement_of_loans_and_guarantees_latest_available_snapshot_09-09-2026.csv
Output path: /home/rigii/ATA/data/processed/ibrd_clean.csv


In [2]:
if not raw_path.exists():
    raise FileNotFoundError(f'Required raw file not found: {raw_path}')

raw_df = pd.read_csv(raw_path, encoding='utf-8-sig', low_memory=False)

if raw_df.columns[0].startswith('\ufeff'):
    raw_df.columns = [col.replace('\ufeff', '') for col in raw_df.columns]

print(f'Raw shape: {raw_df.shape}')
print('\nHead (10 rows):')
display(raw_df.head(10))
print('\nInfo:')
raw_df.info()
print('\nDescribe:')
display(raw_df.describe(include='all').T)


Raw shape: (9518, 35)

Head (10 rows):


,End of Period,Loan Number,Region,Country / Economy Code,Country / Economy,Borrower,Guarantor Country / Economy Code,Guarantor,Loan Type,Loan Status,Interest Rate,Currency of Commitment,Project ID,Project Name,Original Principal Amount (US$),Cancelled Amount (US$),Undisbursed Amount (US$),Disbursed Amount (US$),Repaid to IBRD (US$),Due to IBRD (US$),Exchange Adjustment (US$),Borrower's Obligation (US$),Sold 3rd Party (US$),Repaid 3rd Party (US$),Due 3rd Party (US$),Loans Held (US$),First Repayment Date,Last Repayment Date,Agreement Signing Date,Board Approval Date,Effective Date (Most Recent),Closed Date (Most Recent),Last Disbursement Date,Board approval - Fiscal year,Board approval - Calendar year
0,08/31/2026,IBRD87030,EAST ASIA AND PACIFIC,CN,China,MINISTRY OF FINANCE,CN,China,FSL,Repaying,0.00,NaN,P153173,Anhui Road Maintenance Innovation,"150,000,000.00","12,094,000.66",0.00,"137,905,999.34","23,646,955.00","114,259,044.34",0.00,"114,259,044.34",0.00,0.00,0,"114,259,044.34",05/15/2023,11/15/2036,04/11/2017,02/21/2017,08/17/2017,10/31/2025,04/07/2026,2017,2017
1,08/31/2026,IBRD87040,EAST ASIA AND PACIFIC,CN,China,MINISTRY OF FINANCE,CN,China,FSL,Repaying,0.00,NaN,P153604,Poyang Lake Water Environment Management,"150,000,000.00","14,102,831.83",0.00,"135,897,168.17","6,237,680.03","129,659,488.14",0.00,"129,659,488.14",0.00,0.00,0,"129,659,488.14",09/15/2025,03/15/2042,06/06/2017,03/16/2017,10/13/2017,12/31/2022,05/24/2023,2017,2017
2,08/31/2026,IBRD87200,EAST ASIA AND PACIFIC,CN,China,MINISTRY OF FINANCE,CN,China,FSL,Repaying,0.00,NaN,P154623,China: Gansu TVET Project,"120,000,000.00","1,419,267.45",0.00,"118,580,732.55","16,402,743.76","102,177,988.79",0.00,"102,177,988.79",0.00,0.00,0,"102,177,988.79",03/01/2023,03/01/2047,06/26/2017,03/31/2017,10/20/2017,06/30/2023,12/13/2023,2017,2017
3,08/31/2026,IBRD87440,EAST ASIA AND PACIFIC,CN,China,MINISTRY OF FINANCE,CN,China,FSL,Repaying,0.00,NaN,P154984,China Health Reform Program,"600,000,000.00","1,914,958.33",0.00,"598,006,362.38","82,332,293.28","515,674,069.10","1,511,927.12","517,185,996.22",0.00,0.00,0,"515,674,069.10",10/01/2022,04/01/2051,06/30/2017,05/09/2017,09/11/2017,12/31/2021,08/16/2022,2017,2017
4,08/31/2026,IBRD87660,EAST ASIA AND PACIFIC,CN,China,MINISTRY OF FINANCE,CN,China,FSL,Repaying,0.00,NaN,P153473,Three Gorges Modern Logistics Center,"200,000,000.00","11,095,390.61",0.00,"188,904,609.39","13,563,431.56","175,341,177.83",0.00,"175,341,177.83",0.00,0.00,0,"175,341,177.83",11/01/2023,05/01/2047,09/01/2017,06/30/2017,12/27/2017,12/31/2024,08/21/2025,2017,2017
5,08/31/2026,IBRD87770,EAST ASIA AND PACIFIC,CN,China,MINISTRY OF FINANCE,CN,China,FSL,Repaying,0.00,NaN,P153115,Hunan Integr. Manag of Agri Land Project,"100,000,000.00","2,692,621.81",0.00,"97,307,378.19","14,329,905.34","82,977,472.85",0.00,"82,977,472.85",0.00,0.00,0,"82,977,472.85",12/01/2023,06/01/2043,12/11/2017,08/22/2017,03/07/2018,12/31/2023,06/13/2024,2018,2017
6,08/31/2026,IBRD87910,EAST ASIA AND PACIFIC,CN,China,MINISTRY OF FINANCE,CN,China,FSL,Repaying,0.00,NaN,P154621,China: Guangdong Compulsory Education,"120,000,000.00","2,593,846.07",0.00,"117,406,153.93","12,856,318.89","104,549,835.04",0.00,"104,549,835.04",0.00,0.00,0,"104,549,835.04",05/01/2023,11/01/2042,01/16/2018,10/31/2017,04/02/2018,11/30/2023,05/28/2024,2018,2017
7,08/31/2026,IBRD88000,EAST ASIA AND PACIFIC,CN,China,MINISTRY OF FINANCE,CN,China,FSL,Repaying,0.00,NaN,P147009,Jiangxi Farm Produce Distribution System,"150,000,000.00","3,422,598.99",0.00,"146,577,401.01","23,754,017.03","122,823,383.98",0.00,"122,823,383.98",0.00,0.00,0,"122,823,383.98",06/01/2023,12/01/2041,03/05/2018,12/15/2017,05/15/2018,06/30/2025,12/11/2025,2018,2017
8,08/31/2026,IBRD88370,EAST ASIA AND PACIFIC,CN,China,MINISTRY OF FINANCE,CN,China,FSL,Repaying,0.00,NaN,P158713,Liaoning Safe and Sustainable Urban WS,"250,000,000.00","103,486,796.88",0.00,"146,513,203.12","14,747,939.52","131,765,263.60",0.00,"131,765,263.60",0.00,0.00,0,"131,765,263.60",1


Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9518 entries, 0 to 9517
Data columns (total 35 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   End of Period                     9518 non-null   object 
 1   Loan Number                       9518 non-null   object 
 2   Region                            9518 non-null   object 
 3   Country / Economy Code            9515 non-null   object 
 4   Country / Economy                 9518 non-null   object 
 5   Borrower                          9461 non-null   object 
 6   Guarantor Country / Economy Code  9235 non-null   object 
 7   Guarantor                         9238 non-null   object 
 8   Loan Type                         9518 non-null   object 
 9   Loan Status                       9518 non-null   object 
 10  Interest Rate                     9409 non-null   float64
 11  Currency of Commitment            0 non-null      float64
 12 

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
End of Period,9518,1,08/31/2026,9518,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Loan Number,9518,9518,IBRD87030,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Region,9518,7,LATIN AMERICA AND CARIBBEAN,2975,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country / Economy Code,9515,147,ID,657,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country / Economy,9518,148,Indonesia,657,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Borrower,9461,973,MINISTRY OF FINANCE,1285,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Guarantor Country / Economy Code,9235,126,ID,657,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Guarantor,9238,127,Indonesia,657,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Loan Type,9518,11,FSL,2916,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Loan Status,9518,11,Fully Repaid,6622,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df = raw_df.copy()

date_cols = [
    'End of Period', 'First Repayment Date', 'Last Repayment Date',
    'Agreement Signing Date', 'Board Approval Date',
    'Effective Date (Most Recent)', 'Closed Date (Most Recent)',
    'Last Disbursement Date'
]
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

numeric_cols = [
    'Original Principal Amount (US$)', 'Cancelled Amount (US$)',
    'Undisbursed Amount (US$)', 'Disbursed Amount (US$)',
    'Repaid to IBRD (US$)', 'Due to IBRD (US$)',
    'Exchange Adjustment (US$)', "Borrower's Obligation (US$)",
    'Loans Held (US$)'
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace(',', '', regex=False)
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

text_cols = ['Region', 'Country / Economy', 'Loan Status', 'Loan Type']
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        if col == 'Country / Economy':
            df[col] = df[col].str.title()

print('Date columns converted to datetime:')
print([col for col in date_cols if col in df.columns])
print('\nNumeric columns normalized to float:')
print(numeric_cols)
print('\nText columns standardized:')
print(text_cols)


Date columns converted to datetime:
['End of Period', 'First Repayment Date', 'Last Repayment Date', 'Agreement Signing Date', 'Board Approval Date', 'Effective Date (Most Recent)', 'Closed Date (Most Recent)', 'Last Disbursement Date']

Numeric columns normalized to float:
['Original Principal Amount (US$)', 'Cancelled Amount (US$)', 'Undisbursed Amount (US$)', 'Disbursed Amount (US$)', 'Repaid to IBRD (US$)', 'Due to IBRD (US$)', 'Exchange Adjustment (US$)', "Borrower's Obligation (US$)", 'Loans Held (US$)']

Text columns standardized:
['Region', 'Country / Economy', 'Loan Status', 'Loan Type']


## 1) Derived Feature Engineering

The following fields are created to support repayment analysis, portfolio aging, and risk segmentation.

In [4]:
if 'Board Approval Date' in df.columns and 'End of Period' in df.columns:
    df['loan_age_days'] = (df['End of Period'] - df['Board Approval Date']).dt.days
    df['loan_age_years'] = df['loan_age_days'] / 365.25
else:
    df['loan_age_days'] = np.nan
    df['loan_age_years'] = np.nan

original = df.get('Original Principal Amount (US$)', pd.Series(0.0, index=df.index)).fillna(0.0)
cancelled = df.get('Cancelled Amount (US$)', pd.Series(0.0, index=df.index)).fillna(0.0)
disbursed = df.get('Disbursed Amount (US$)', pd.Series(0.0, index=df.index)).fillna(0.0)
undisbursed = df.get('Undisbursed Amount (US$)', pd.Series(0.0, index=df.index)).fillna(0.0)
repaid = df.get('Repaid to IBRD (US$)', pd.Series(0.0, index=df.index)).fillna(0.0)
due = df.get('Due to IBRD (US$)', pd.Series(0.0, index=df.index)).fillna(0.0)

df['disbursed_percent'] = np.divide(
    disbursed, original.replace(0, np.nan), out=np.zeros(len(df), dtype=float), where=original.ne(0)
) * 100
df['cancelled_percent'] = np.divide(
    cancelled, original.replace(0, np.nan), out=np.zeros(len(df), dtype=float), where=original.ne(0)
) * 100
df['undisbursed_percent'] = np.divide(
    undisbursed, original.replace(0, np.nan), out=np.zeros(len(df), dtype=float), where=original.ne(0)
) * 100
df['repayment_ratio'] = np.divide(
    repaid, disbursed.replace(0, np.nan), out=np.zeros(len(df), dtype=float), where=disbursed.ne(0)
).clip(0, 1)
df['outstanding_ratio'] = np.divide(
    due, disbursed.replace(0, np.nan), out=np.zeros(len(df), dtype=float), where=disbursed.ne(0)
).clip(0, 1)
df['net_disbursed'] = disbursed - cancelled

status_values = df.get('Loan Status', pd.Series('', index=df.index)).fillna('').astype(str).str.strip()
df['is_active'] = status_values.isin(['Disbursing', 'Disbursing&Repaying', 'Repaying', 'Effective'])
df['is_fully_repaid'] = status_values.eq('Fully Repaid')
df['is_cancelled'] = status_values.isin(['Fully Cancelled', 'Cancelled'])

if 'Closed Date (Most Recent)' in df.columns:
    df['is_closed'] = df['Closed Date (Most Recent)'].notna()
else:
    df['is_closed'] = False

loan_type_values = df.get('Loan Type', pd.Series('', index=df.index)).fillna('').astype(str).str.upper()
df['is_guarantee'] = loan_type_values.str.contains('GURB|GUBF|BLNR', regex=True, na=False)

print('Derived metrics created:')
for col in ['loan_age_days', 'loan_age_years', 'disbursed_percent', 'cancelled_percent', 'undisbursed_percent', 'repayment_ratio', 'outstanding_ratio', 'net_disbursed', 'is_active', 'is_fully_repaid', 'is_cancelled', 'is_closed', 'is_guarantee']:
    print(col)


Derived metrics created:
loan_age_days
loan_age_years
disbursed_percent
cancelled_percent
undisbursed_percent
repayment_ratio
outstanding_ratio
net_disbursed
is_active
is_fully_repaid
is_cancelled
is_closed
is_guarantee


In [5]:
original_usd = df.get('Original Principal Amount (US$)', pd.Series(0.0, index=df.index)).fillna(0.0)
df['loan_size_category'] = pd.cut(
    original_usd,
    bins=[-1, 50_000_000, 200_000_000, 500_000_000, np.inf],
    labels=['Small', 'Medium', 'Large', 'Mega'],
    right=False
)

df['age_category'] = pd.cut(
    df['loan_age_years'].fillna(0),
    bins=[-1, 5, 10, 20, 30, 50, np.inf],
    labels=['0-5', '5-10', '10-20', '20-30', '30-50', '50+'],
    right=False
)

if 'Board Approval Date' in df.columns:
    df['approval_year'] = df['Board Approval Date'].dt.year
    df['approval_decade'] = (df['approval_year'] // 10) * 10
else:
    df['approval_year'] = np.nan
    df['approval_decade'] = np.nan

def classify_portfolio(row):
    if row.get('is_cancelled', False):
        return 'Cancelled'
    if row.get('is_fully_repaid', False):
        return 'Fully Repaid'
    if row.get('repayment_ratio', 0) > 0.9:
        return 'Near Completion'
    if row.get('repayment_ratio', 0) > 0.5:
        return 'Partially Repaid'
    return 'Early Stage'

df['portfolio_status'] = df.apply(classify_portfolio, axis=1)

print('Categorization fields created:')
for col in ['loan_size_category', 'age_category', 'approval_year', 'approval_decade', 'portfolio_status']:
    print(col)


Categorization fields created:
loan_size_category
age_category
approval_year
approval_decade
portfolio_status


In [6]:
def calculate_risk_score(row):
    status = str(row.get('Loan Status', '')).strip()
    score = 0.0

    if status in ['Cancelled', 'Fully Cancelled']:
        score += 0.65
    elif status == 'Fully Repaid':
        return 0.0

    repayment_ratio = float(row.get('repayment_ratio', 0.0) or 0.0)
    outstanding_ratio = float(row.get('outstanding_ratio', 0.0) or 0.0)
    loan_age_years = float(row.get('loan_age_years', 0.0) or 0.0)
    cancelled_pct = float(row.get('cancelled_percent', 0.0) or 0.0)
    undisbursed_pct = float(row.get('undisbursed_percent', 0.0) or 0.0)

    if repayment_ratio < 0.35:
        score += 0.40
    elif repayment_ratio < 0.75:
        score += 0.20
    else:
        score += 0.05

    if outstanding_ratio > 0.5:
        score += 0.20
    elif outstanding_ratio > 0.25:
        score += 0.10

    if loan_age_years > 30:
        score += 0.10
    elif loan_age_years > 15:
        score += 0.05

    if cancelled_pct > 25:
        score += 0.10
    if undisbursed_pct > 80:
        score += 0.10
    if row.get('is_active', False):
        score += 0.05

    return float(np.clip(score, 0.0, 1.0))

df['risk_score'] = df.apply(calculate_risk_score, axis=1)
print('Risk score calculated for each row.')
display(df[['Loan Number', 'Loan Status', 'repayment_ratio', 'risk_score']].head(10))


Risk score calculated for each row.


,Loan Number,Loan Status,repayment_ratio,risk_score
0,IBRD87030,Repaying,0.17,0.65
1,IBRD87040,Repaying,0.05,0.65
2,IBRD87200,Repaying,0.14,0.65
3,IBRD87440,Repaying,0.14,0.65
4,IBRD87660,Repaying,0.07,0.65
5,IBRD87770,Repaying,0.15,0.65
6,IBRD87910,Repaying,0.11,0.65
7,IBRD88000,Repaying,0.16,0.65
8,IBRD88370,Repaying,0.10,0.75
9,IBRD88460,Repaying,1.00,0.10


## 2) Engineered Feature Dictionary

- `loan_age_days`: Age of the loan in days from approval to the reporting period. Business purpose: distinguishes mature loans from new disbursements in aging analysis.
- `loan_age_years`: Same age normalized to years for comparability across time horizons. Business purpose: supports lifecycle and vintage segmentation.
- `disbursed_percent`: Share of original principal actually disbursed. Business purpose: measures implementation progress and commitment utilization.
- `cancelled_percent`: Share of principal cancelled relative to original commitment. Business purpose: flags contract amendments or project under-delivery.
- `undisbursed_percent`: Share remaining undisbursed. Business purpose: highlights uncommitted balances and pipeline utilization.
- `repayment_ratio`: Repaid amount divided by disbursed amount, clipped to a 0–1 range. Business purpose: indicates how much of the disbursed capital has already been repaid.
- `outstanding_ratio`: Due amount divided by disbursed amount, clipped to a 0–1 range. Business purpose: captures current arrears intensity versus total disbursement.
- `net_disbursed`: Disbursed amount minus cancellations. Business purpose: estimates the effective amount actually utilized by the borrower.
- `is_active`: Flag for currently active status categories. Business purpose: separates operational lending from fully repaid or cancelled loans.
- `is_fully_repaid`: Flag for a fully repaid status. Business purpose: identifies completed loans for portfolio turnover analysis.
- `is_cancelled`: Flag for cancelled or fully cancelled loans. Business purpose: isolates non-performing or terminated exposures.
- `is_guarantee`: Flag for guarantee-related loan types. Business purpose: highlights guaranteed or contingent financing structures.
- `loan_size_category`: Commitment bucket based on original principal. Business purpose: supports size-based portfolio segmentation and risk comparison.
- `age_category`: Life-cycle bucket based on years since approval. Business purpose: helps compare early-stage versus mature loans.
- `approval_year` / `approval_decade`: Approval vintage features. Business purpose: aids historical trend and cohort analysis.
- `portfolio_status`: Rule-based portfolio state. Business purpose: classifies each loan into management states like Early Stage, Partially Repaid, and Near Completion.
- `risk_score`: Rule-based risk measure combining repayment, outstanding exposure, age, and cancellation patterns. Business purpose: provides a simple, explainable risk indicator for watchlist prioritization.

In [7]:
amount_cols = [
    'Original Principal Amount (US$)', 'Cancelled Amount (US$)', 'Undisbursed Amount (US$)',
    'Disbursed Amount (US$)', 'Repaid to IBRD (US$)', 'Due to IBRD (US$)',
    'Exchange Adjustment (US$)', "Borrower's Obligation (US$)", 'Loans Held (US$)'
]

for col in amount_cols:
    if col in df.columns:
        assert (pd.to_numeric(df[col], errors='coerce') < 0).sum() == 0, f'Negative values found in {col}'

assert df['repayment_ratio'].between(0, 1).all(), 'repayment_ratio contains values outside [0, 1]'

print(f'Final data shape: {df.shape}')
print('\nFinal head (10):')
display(df.head(10))


AssertionError: Negative values found in Undisbursed Amount (US$)

In [ ]:
raw_columns = set(raw_df.columns)
new_columns = [c for c in df.columns if c not in raw_columns]

df.to_csv(processed_path, index=False)
print(f'Cleaned data saved to: {processed_path}')
print(f'New columns created: {len(new_columns)}')
print(new_columns)
print('\nSaved file preview:')
saved_df = pd.read_csv(processed_path, low_memory=False)
print(saved_df.shape)
display(saved_df.head(5))
